In [ ]:
# change in loaddata.py to "augmented_maps2d/gaussian_alpha_005"
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the parent directory to the path so we can import Danial/Alen's code
sys.path.append(os.path.abspath(".."))

from train_single import get_model_class
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from loaddata import load_level_data

def evaluate_model_on_folder(model_name, level, data_root):
    """Loads a pre-trained model and evaluates it using proper preprocessing."""
    
    # 1. Load the Data using the team's official pipeline
    # This guarantees we get the exact same log10 and Mean/Std preprocessing!
    # We ignore X_train/Y_train and just take the validation set (X_val, Y_val).
    _, _, X_val, Y_val = load_level_data(level=level, val_fold=5, root=data_root)
    
    # Convert to PyTorch tensors
    x_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(Y_val, dtype=torch.float32).to(device)
    
    # 2. Load the Pre-trained Model
    model_class = get_model_class(model_name)
    model = model_class().to(device)
    
    weights_path = f"../checkpoints/{model_name}_level{level}.pth"
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
    else:
        print(f"Warning: No weights found at {weights_path}. Evaluating untrained model!")
        
    model.eval()
    
    # 3. Predict and Calculate Statistics
    criterion = nn.MSELoss()
    with torch.no_grad():
        predictions = model(x_tensor)
        
        total_mse = criterion(predictions, y_tensor).item()
        
        # Calculate MAE (Mean Absolute Error) for Omega_m and sigma_8 separately
        mae_omega_m = torch.mean(torch.abs(predictions[:, 0] - y_tensor[:, 0])).item()
        mae_sigma_8 = torch.mean(torch.abs(predictions[:, 1] - y_tensor[:, 1])).item()

    return {
        "Total MSE": total_mse,
        "Omega_m MAE": mae_omega_m,
        "Sigma_8 MAE": mae_sigma_8
    }

In [ ]:
# Configuration
MODEL_TO_TEST = "model_10_5"  # Use one of the names from the JSON files you showed me
LEVEL = 3                     

# The directories we want to compare
directories_to_test = {
    "Original (Clean)": "maps2d",
    "Gaussian Noise (1%)": "augmented_maps2d/gaussian_alpha_001",
    "Gaussian Noise (5%)": "augmented_maps2d/gaussian_alpha_005"
}

results = {}

for label, folder in directories_to_test.items():
    print(f"\n--- Testing on: {label} ---")
    try:
        metrics = evaluate_model_on_folder(MODEL_TO_TEST, LEVEL, folder)
        results[label] = metrics
    except Exception as e:
        print(f"Error loading {folder}: {e}")

# Convert to a beautiful Pandas DataFrame
df_results = pd.DataFrame(results).T
display(df_results)

In [ ]:
# Plotting the degradation in performance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot Omega_m error
df_results['Omega_m MAE'].plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title(f"Error in Predicting $\Omega_m$ ({MODEL_TO_TEST})")
axes[0].set_ylabel("Mean Absolute Error (Lower is better)")
axes[0].tick_params(axis='x', rotation=0)

# Plot Sigma_8 error
df_results['Sigma_8 MAE'].plot(kind='bar', ax=axes[1], color='salmon', edgecolor='black')
axes[1].set_title(f"Error in Predicting $\sigma_8$ ({MODEL_TO_TEST})")
axes[1].set_ylabel("Mean Absolute Error (Lower is better)")
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()